# Tutorial 13-1: The Architect – "Building a Mini-LLaVA"

**Course:** CSEN 342: Deep Learning  
**Topic:** Multimodal LLMs, Vision Encoders, and Projection Layers

## Objective
How do Multimodal LLMs (like GPT-4V or LLaVA) see images? They don't have "eyes" in the traditional sense. Instead, they rely on a **Vision Encoder** to turn pixels into vectors, and a **Projection Layer** to translate those vectors into the "language" of the LLM.

In this tutorial, we will build a **Mini-LLaVA** from scratch using open-source components:
1.  **Vision Encoder:** CLIP (ViT-Base).
2.  **Language Model:** TinyLlama (1.1B parameters).
3.  **The Bridge:** A trainable Linear Projection layer.

We will wire them together and train the model on a single example to prove that the gradients flow correctly from text to image.

*Note*: Run this notebook under the `Transformers Bundle`.

---

## Part 1: Loading the Components

We need `transformers` to load our pre-trained models. We use **TinyLlama** because it is small enough to run easily on limited hardware while still being a fully functional LLM.

In [1]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
import requests
from PIL import Image
from io import BytesIO

# Config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vision_model_name = "openai/clip-vit-base-patch32"
llm_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading models... (This may take a minute)")

# 1. Load Vision Encoder (CLIP)
# We only need the 'vision_model' part of CLIP, not the text part.
clip_model = AutoModel.from_pretrained(vision_model_name)
vision_encoder = clip_model.vision_model.to(device)
from transformers import AutoProcessor
clip_processor = AutoProcessor.from_pretrained(vision_model_name)

# 2. Load LLM (TinyLlama)
tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm = AutoModelForCausalLM.from_pretrained(llm_model_name).to(device)

# Freeze the pre-trained models (We only want to train the bridge)
for param in vision_encoder.parameters():
    param.requires_grad = False
for param in llm.parameters():
    param.requires_grad = False

print("Models loaded and frozen.")

Loading models... (This may take a minute)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Models loaded and frozen.


---

## Part 2: The Architecture (The Bridge)

CLIP outputs embeddings of size **768**. TinyLlama expects embeddings of size **2048**.
We need a **Projection Layer** to translate between them.

The Data Flow:
1.  `Image` $\to$ `Vision Encoder` $\to$ `Image Embeddings` (Batch, 50, 768)
2.  `Image Embeddings` $\to$ `Projection Layer` $\to$ `Projected Embeddings` (Batch, 50, 2048)
3.  `Text Tokens` $\to$ `LLM Embeddings` (Batch, Seq, 2048)
4.  Concatenate `[Projected Image, Text]` $\to$ `LLM`



In [2]:
class MiniLLaVA(nn.Module):
    def __init__(self, vision_encoder, llm, vision_hidden_size=768, llm_hidden_size=2048):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.llm = llm
        
        # The "Bridge": A simple Linear layer (Projector)
        # In real LLaVA, this is often a 2-layer MLP, but Linear works for demo.
        self.projector = nn.Linear(vision_hidden_size, llm_hidden_size)
        
    def forward(self, pixel_values, input_ids, attention_mask=None, labels=None):
        # 1. Get Image Embeddings
        # Output shape: (Batch, Num_Patches, Vision_Dim)
        with torch.no_grad():
            vision_outputs = self.vision_encoder(pixel_values=pixel_values)
            image_embeds = vision_outputs.last_hidden_state
            
        # 2. Project to LLM dimension
        # Shape: (Batch, Num_Patches, LLM_Dim)
        image_embeds_projected = self.projector(image_embeds)
        
        # 3. Get Text Embeddings
        # We use the LLM's own embedding layer to turn token IDs into vectors
        # Shape: (Batch, Seq_Len, LLM_Dim)
        text_embeds = self.llm.model.embed_tokens(input_ids)
        
        # 4. Fuse (Concatenate)
        # We prepend the image tokens to the text tokens
        # Final Shape: (Batch, Num_Patches + Seq_Len, LLM_Dim)
        inputs_embeds = torch.cat([image_embeds_projected, text_embeds], dim=1)
        
        # 5. Pass through LLM
        # We need to adjust the attention mask to account for the image tokens
        if attention_mask is not None:
            # Add 1s for the image tokens
            image_mask = torch.ones(
                image_embeds.shape[:2], dtype=torch.long, device=image_embeds.device
            )
            attention_mask = torch.cat([image_mask, attention_mask], dim=1)

        outputs = self.llm(
            inputs_embeds=inputs_embeds, 
            attention_mask=attention_mask,
            # We don't calculate loss on the image tokens, so we would technically
            # pad the labels. For simplicity in this demo, we assume the LLM handles it
            # or we are doing generation.
        )
        
        return outputs

# Initialize our custom model
model = MiniLLaVA(vision_encoder, llm).to(device)
print("MiniLLaVA assembled!")

MiniLLaVA assembled!


---

## Part 3: Data Preparation (The Sanity Check)

We don't have a dataset loaded. To prove this works, we will "overfit" a single example. We will take an image of a cat and train the model to output "This is a photo of a cat."

In [3]:
# 1. Get an Image
url = "http://images.cocodataset.org/val2017/000000039769.jpg" # A classic cat image
response = requests.get(url)
image = Image.open(BytesIO(response.content))

# 2. Prepare Inputs
text = "User: Describe this image. Assistant: This is a photo of a cat."
inputs = clip_processor(images=image, return_tensors="pt").to(device)
text_inputs = tokenizer(text, return_tensors="pt").to(device)

# 3. Prepare Labels
# In Causal LM training, labels = input_ids. The model predicts the next token.
labels = text_inputs.input_ids.clone()

print("Data Ready.")
print(f"Image Input Shape: {inputs['pixel_values'].shape}")
print(f"Text Input Shape: {text_inputs['input_ids'].shape}")

Data Ready.
Image Input Shape: torch.Size([1, 3, 224, 224])
Text Input Shape: torch.Size([1, 19])


---

## Part 4: Training the Bridge

We will train **only** the projection layer. The Vision Encoder and LLM remain frozen. We expect the loss to drop quickly as the projection layer learns to map the "Cat" image features to the "Cat" text tokens.

In [4]:
optimizer = torch.optim.Adam(model.projector.parameters(), lr=1e-3)

print("Starting 'Sanity Check' Training...")

# Create labels that account for the image shift
# The LLM sees [Image Tokens] + [Text Tokens]. 
# We need to predict [Text Tokens]. The Image Tokens are just context (labels=-100).
# Number of image tokens = 50 (for CLIP-Base patch32)
num_image_tokens = 50
dummy_image_labels = torch.full((1, num_image_tokens), -100).to(device)
full_labels = torch.cat([dummy_image_labels, labels], dim=1)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    # Note: We need to manipulate the forward pass slightly for standard CausalLM loss calculation
    # usually handled by the HF Trainer. Here we manually construct inputs_embeds.
    # For simplicity in this manual loop, we will just call the model and extract logits.
    
    outputs = model(
        pixel_values=inputs['pixel_values'],
        input_ids=text_inputs['input_ids'],
        attention_mask=text_inputs['attention_mask']
    )
    
    # Calculate Loss
    # Shift logits so token < n predicts token < n+1
    logits = outputs.logits
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = full_labels[..., 1:].contiguous()
    
    loss_fct = nn.CrossEntropyLoss()
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss {loss.item():.4f}")
        if loss.item() < 0.1:
            print("Converged!")
            break

Starting 'Sanity Check' Training...
Epoch 0: Loss 11.0605
Epoch 10: Loss 2.5466
Epoch 20: Loss 1.0901
Epoch 30: Loss 0.3943
Epoch 40: Loss 0.2831
Epoch 50: Loss 0.2275
Epoch 60: Loss 0.1796
Epoch 70: Loss 0.1380
Epoch 80: Loss 0.1033
Epoch 90: Loss 0.0758
Converged!


### Conclusion

You have successfully built a Multimodal LLM architecture.

**What happens inside?**
1.  CLIP extracts abstract visual features (lines, shapes, "cat-ness").
2.  Your **Projection Layer** learned to rotate/scale those features so they look like the word embedding for "Cat" to the LLM.
3.  The LLM attends to those "visual words" and generates the description.

In a real application, you would train this on millions of image-text pairs (like LLaVA-Instruct) so the projection layer learns a general mapping, not just for one cat.